# Hands-On AI Assistant Workshop: From Autocomplete to Autonomy
**Symposium**: EESTEC LC Thessaloniki AI Symposium  
**Instructor**: Themistoklis Diamantopoulos  
**Stack**: LangChain, FastMCP, Chainlit, and Google Gemini  

Welcome to the hands-on portion of the workshop! This Google Colab notebook is completely self-contained.
We will progress step-by-step chronologically through all 5 Phases of building an AI Assistant.

### How this Interactive Workshop is Structured:
1. **Git Repository standard**: We clone the unified repository at the start. All code bases are run directly from modular repository files.
2. **Global detached Tunnel**: We launch a single, secure Cloudflare tunnel at the start. This public link will serve as your portal for both Phase 3 and Phase 5 web interfaces!
3. **Dynamic custom Sandboxing**: You will write your custom system persona and edit/add tools directly inside the notebook cells, allowing you to test different combinations of instructions and capabilities dynamically!

## Step 0: Clone Workspace Codebase
Let's clone the complete Git repository of the workshop to populate our Colab workspace with the unified components.

### File Structure of the Repository:
- `hands_on_colab.ipynb`: This interactive notebook.
- `model.py`: The unified model initialization layer (Google Gemini/OpenAI/Anthropic).
- `main1.py`: Phase 1 - Stateless console CLI chatbot.
- `main2.py`: Phase 2 - Live streaming console CLI chatbot.
- `main3_chainlit.py`: Phase 3 - Clean web UI transition with Chainlit.
- `system_prompt.txt`: Custom system persona instructions (written on-demand).
- `mcpserver.py`: Phase 4 - Simple Model Context Protocol server (geocoding, weather, travel notes).
- `main4_agent_chainlit.py`: Phase 5 - Parallel resilient agent with tool binding.
- `launch_tunnel.py`: Helper script to manage global Cloudflare port exposure.
- `mcp_simple.py`, `mcp_weather.py`, `mcp_journal.py`, `mcp_advanced.py`: Modular example tool servers.
- `system_prompt_simple.txt`, `system_prompt_weather.txt`, `system_prompt_journal.txt`, `system_prompt_advanced.txt`: Modular system prompt combinations.

In [ ]:
# Clone the workshop repository and enter the directory
!git clone https://github.com/AuthEceSoftEng/tutorial-ai-assistant.git
%cd tutorial-ai-assistant

## Step 1: Installation of Dependencies and Tunneling Utilities
We install all required libraries and download the Cloudflare Quick Tunnel binary immediately at the start to ensure our portals are ready.

In [ ]:
# 1. Install pip libraries
print("Installing dependencies...")
!pip install -q langchain langchain-mcp-adapters fastmcp chainlit requests python-dotenv langchain-google-genai

# 2. Download Cloudflare Tunnel binary
import subprocess, os
if not os.path.exists("cloudflared"):
    print("Downloading Cloudflare Tunnel...")
    subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", "cloudflared"])
    subprocess.run(["chmod", "+x", "cloudflared"])
    print("Cloudflare Tunnel ready.\n")
else:
    print("Cloudflare Tunnel already exists.")

## Step 2: Configure API Key Credentials
To build your AI assistant, you will need a Google Gemini API Key. 

### 🔑 Get Your Key:
Generate a free API Key in Google AI Studio: **[Google AI Studio](https://aistudio.google.com/)**  

*(Note: We use the ultra-fast, capable **Gemini 2.5 Flash** model for this hands-on lab. Running this cell automatically writes your settings to `.env`.)*

In [ ]:
#@title Configure Gemini API Key
#@markdown Get your free key at: https://aistudio.google.com/
LLM_API_KEY = "PASTE_YOUR_API_KEY_HERE" #@param {type:"string"}

with open(".env", "w") as f:
    f.write(f"LLM_PROVIDER=google\n")
    f.write(f"LLM_MODEL=gemini-2.5-flash\n")
    f.write(f"LLM_API_KEY={LLM_API_KEY}\n")

print(".env configuration file successfully written!")

## Step 3: Launch Global Port Exposer Tunnel
We launch our Cloudflare Quick Tunnel on Port 8000 at the very start.
**This cell will output your secure public link (trycloudflare.com). Keep this link open!** 
Whenever you run Phase 3 or Phase 5 in the later cells, that active web server will automatically be exposed on this exact link!

*(Note: This cell is highly resilient. If you rerun it, it will detect the existing running tunnel and display your original URL!)*

In [ ]:
# Launch global Cloudflare tunnel in detached background session
!python3 launch_tunnel.py

## Phase 1: Stateless CLI Chatbot
In this phase, we build a basic terminal prompt loop chatbot.
API endpoints do not store history. Memory is managed client-side using simple message arrays (`SystemMessage`, `HumanMessage`, `AIMessage`).

**Run the code cell below directly inside Colab to start your chatbot! Type `exit` to stop.**

In [ ]:
print("Initializing stateless AI Assistant... (Loading LangChain and Gemini configurations)", flush=True)
from langchain_core.messages import SystemMessage, HumanMessage
from model import get_model

aimodel = get_model()
system_prompt = SystemMessage(content="You are a helpful AI assistant. Your tone is professional yet friendly.")

print("AI Assistant is ready! (Type 'exit' to stop)")

messages = [system_prompt]

while True:
    user_input = input("\nYou: ")
    if user_input.lower() in ["exit", "quit"]:
        break
    
    messages.append(HumanMessage(content=user_input))
    
    # Get response
    response = aimodel.invoke(messages)  # @UndefinedVariable
    
    print(f"\nAI: {response.content}")
    
    # Update history for context
    messages.append(response)


## Phase 2: CLI Chatbot with Token-by-Token Streaming
We resolve the latency bottleneck. We swap `.invoke()` with `.stream()`, iterating over token chunks and immediately flushing characters to the screen.

**Run the code cell below directly inside Colab to experience live token streaming! Type `exit` to stop.**

In [ ]:
print("Initializing streaming AI Assistant... (Loading generator configurations)", flush=True)
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from model import get_model

aimodel = get_model()

import os
if os.path.exists("system_prompt.txt"):
    with open("system_prompt.txt", "r") as infile:
        system_prompt_text = infile.read().strip()
else:
    system_prompt_text = "You are a helpful AI assistant. Your tone is professional yet friendly."
system_prompt = SystemMessage(content=system_prompt_text)

print("AI Assistant is ready! (Type 'exit' to stop)")

messages = [system_prompt]

while True:
    user_input = input("\nYou: ")
    if user_input.lower() in ["exit", "quit"]:
        break
    
    messages.append(HumanMessage(content=user_input))
    
    # Get response
    print("AI: ", end="", flush=True) # Print prefix without a newline
    full_response = "" # Use .stream() instead of .invoke()
    for chunk in aimodel.stream(messages):  # @UndefinedVariable
        content = chunk.content
        print(content, end="", flush=True) # Print tokens as they arrive
        full_response += content
    print() # Add a newline at the end
    
    # Update history for context
    messages.append(AIMessage(content=full_response))


## Phase 3: Transition to Chainlit Web UI
We transition from CLI to browser chat using **Chainlit**—managing async callbacks (`on_chat_start`, `on_message`) and session storage.

**Run the cell below to launch Phase 3, then refresh/open the master trycloudflare URL from Step 3 to interact! Click Stop to terminate when done.**

In [ ]:
# Start stateless Chainlit app in the host on Port 8000
import os
if os.path.exists("tunnel_url.txt"):
    with open("tunnel_url.txt", "r") as f:
        url = f.read().strip()
    print("=======================================================")
    print(f"YOUR CHAINLIT CHAT IS LIVE! Access it here:\n{url}")
    print("=======================================================")

!chainlit run main3_chainlit.py --host 0.0.0.0 --port 8000

## Step 5: System Prompt Persona Configuration
Now, **you will author your assistant's core persona guidelines from scratch**!
Use the editable template cell below to write your custom rules into `system_prompt.txt`.

### 🧠 Prompt & Tool Combinations Sandbox:
You can dynamically pair different steering instructions with different tool servers to change the agent's behavior entirely! We have packaged **4 visual combinations** in your workspace:
1. **Math & Clock (Default)**: Write the rules below, and combine with the simple `mcpserver.py` tool. (Template: `system_prompt_simple.txt`)
2. **Weather Forecaster**: Copy `system_prompt_weather.txt` contents into `system_prompt.txt` and combine with the `mcp_weather.py` tool.
3. **Travel Architect**: Copy `system_prompt_journal.txt` contents into `system_prompt.txt` and combine with the `mcp_journal.py` tool.
4. **Developer Co-Pilot**: Copy `system_prompt_advanced.txt` contents into `system_prompt.txt` and combine with the `mcp_advanced.py` tool.

In [ ]:
%%writefile system_prompt.txt
You are a helpful and professional AI assistant.
Your tone is polite, concise, and academic.

## Phase 4: Exposing Local Tools via Model Context Protocol (MCP)
The Model Context Protocol standardizes how LLMs interact with local APIs and tools.
Instead of hardcoding custom tool calls, we launch a local lightweight **FastMCP** server (`mcpserver.py`) that handles custom python calculations.

### Modular Example Servers in Your Workspace:
*   `mcp_simple.py`: Contains exactly **2 simple tools** (`add_numbers`, `get_time`). Perfect for baseline testing.
*   `mcp_weather.py`: Exposes open-meteo weather geocoding tools.
*   `mcp_journal.py`: Logs travel notes to local text storage.
*   `mcp_advanced.py`: Reference developer utility server (DuckDuckGo search, AST linter, Git status, docs compiler).

**Modify, add, or remove tools directly in the cell below, then run it to write `mcpserver.py` to disk!**

In [ ]:
%%writefile mcpserver.py
# pip install fastmcp
from fastmcp import FastMCP
from datetime import datetime

# Initialize the server
mcp = FastMCP("MyAssistantTools")

@mcp.tool()
def add_numbers(a: int, b: int) -> int:
    """A simple tool to add two numbers together."""
    return a + b

@mcp.tool()
def get_time() -> dict:
    """Returns the current time with timezone offset."""
    local_dt = datetime.now().astimezone()
    return {
        "current_time": local_dt.strftime("%Y-%m-%d %H:%M:%S"),
        "timezone": local_dt.strftime("%Z"),
        "utc_offset": local_dt.strftime("%z")
    }

# =========================================================
# STUDENTS: ADD, EDIT, OR REMOVE YOUR CUSTOM MCP TOOLS BELOW:
# =========================================================


if __name__ == "__main__":
    mcp.run(transport="http", host="127.0.0.1", port=8001)

## Phase 5: Parallel Resilient Agent
We bind the MCP tools to our LLM. Our agent runs tool calls concurrently (`asyncio.gather()`), catches exceptions defensively, and streams pre-computed answers instantly.

*(Note: The Parallel Agent starts your custom tools server `mcpserver.py` in the background automatically!)*

**Run the cell below to start the final Agent on Port 8000. Open/refresh your master trycloudflare URL from Step 3 to interact!**

In [ ]:
# Start Parallel Agent Chainlit app on Port 8000
import os
if os.path.exists("tunnel_url.txt"):
    with open("tunnel_url.txt", "r") as f:
        url = f.read().strip()
    print("=======================================================")
    print(f"YOUR AGENT IS LIVE! Access it here:\n{url}")
    print("=======================================================")

!chainlit run main4_agent_chainlit.py --host 0.0.0.0 --port 8000